# ConcreteAI — Concrete Compressive Strength Prediction

Dataset UCI Yeh (1998), DOI [10.24432/C5PK67](https://doi.org/10.24432/C5PK67). Jalankan dari root repository setelah dataset diunduh.

## 1. Objective
Memprediksi kuat tekan beton (MPa) dari tujuh komponen campuran dan umur beton. Baseline: Linear Regression. Comparison: Random Forest.

In [ ]:
from pathlib import Path
import pandas as pd
from src.train import load_and_validate, split_by_mixture, FEATURES, TARGET

ROOT = Path.cwd()
df = load_and_validate(ROOT / 'data' / 'concrete_strength.csv')
df.head()

## 2. Data validation and EDA
Duplicate rows dilaporkan, bukan dihapus otomatis. EDA dan grafik lengkap dihasilkan oleh `python src/train.py`.

In [ ]:
print('Shape:', df.shape)
print('Missing values:', int(df.isna().sum().sum()))
print('Exact duplicate rows:', int(df.duplicated().sum()))
print('Repeated mixture-age rows:', int(df.duplicated(subset=FEATURES).sum()))
print('Unique mixtures:', df[FEATURES[:-1]].drop_duplicates().shape[0])
df.describe().T

## 3. Group-aware train/test split
Group key memakai tujuh komponen material (tanpa age), sehingga komposisi campuran yang sama tidak tersebar ke train dan test.

In [ ]:
train_idx, test_idx, groups = split_by_mixture(df)
print('Train rows:', len(train_idx), 'Test rows:', len(test_idx))
print('Train mixtures:', len(set(groups[train_idx])), 'Test mixtures:', len(set(groups[test_idx])))
assert set(groups[train_idx]).isdisjoint(set(groups[test_idx]))

## 4. Train and evaluate
Jalankan `python src/train.py` dari root repository untuk menghasilkan model, MAE, RMSE, R², grouped cross-validation, predictions, coefficients dan plots. Jangan menulis angka sebelum file hasil eksekusi tersedia.

In [ ]:
from src.train import main
# Uncomment to execute the full training pipeline:
# main()

## 5. Mathematical notes
Linear Regression meminimalkan Sum of Squared Errors (Ordinary Least Squares). Random Forest merata-ratakan prediksi decision trees yang dilatih menggunakan bootstrap samples dan randomized feature subsets. Feature importance bersifat deskriptif, bukan kausal.

## 6. Limitations
Dataset historis dan ukuran sampel membatasi generalisasi. Prediksi tidak menggantikan pengujian laboratorium, standar material, structural design atau keputusan keselamatan konstruksi.